In [1]:
import sys
from pathlib import Path
import torch
import torch.nn as nn
import torch.nn.functional as F
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

sys.path.append('../reconstruction')
sys.path.append('./reconstruction')
from util import Population

from standardize_align_new import Standardizer, SingleTransformerEncoderStandardizer

from models import base_CNN, two_CNN, var_CNN, base_RNN, var_RNN, base_TransformerEncoder

generalized align


In [2]:
t = torch.randn(3, 4)
print(t)
sorted, indeces = t.sort()
print(sorted)
print(indeces)

tensor([[-0.5530, -0.7528,  1.1806,  0.0207],
        [-0.3454,  1.3735, -0.9262, -0.6513],
        [-0.8834, -0.4236, -0.0978,  1.8544]])
tensor([[-0.7528, -0.5530,  0.0207,  1.1806],
        [-0.9262, -0.6513, -0.3454,  1.3735],
        [-0.8834, -0.4236, -0.0978,  1.8544]])
tensor([[1, 0, 3, 2],
        [2, 3, 0, 1],
        [0, 1, 2, 3]])


In [2]:
def print_weights(model):
    for name, param in model.named_parameters():
        if 'weight' in name:
            print(f'Layer: {name} - Weights')
            print(param)
        elif 'bias' in name:
            print(f'Layer: {name} - Biases')
            print(param)

def print_pairs(model, blackbox):
    for (name1, param1), (name2, param2) in zip(model.named_parameters(), blackbox.named_parameters()):
        if 'weight' in name1:
            print(f'Layer: {name1} - Reconstructed Weights')
            print(param1)
            print(f'Layer: {name2} - Blackbox Weights')
            print(param2)
        elif 'bias' in name1:
            print(f'Layer: {name1} - Reconstructed Biases')
            print(param1)
            print(f'Layer: {name2} - Blackbox Biases')
            print(param2)

In [3]:
def print_weights(model):
    for name, param in model.named_parameters():
        if 'weight' in name:
            print(f'Layer: {name} - Weights')
            print(param)
        elif 'bias' in name:
            print(f'Layer: {name} - Biases')
            print(param)

In [4]:
def print_sorted_abs_weights(model):
    for name, param in model.named_parameters():
        if 'weight' in name:
            print(f'Layer: {name} - Weights')
            sorted = torch.abs(param)
            sorted, _ = torch.sort(sorted)
            print(sorted)
        elif 'bias' in name:
            print(f'Layer: {name} - Biases')
            sorted = torch.abs(param)
            sorted, _ = torch.sort(sorted)
            print(sorted)

In [5]:
def print_diff(model1, model2):
	for m1np, m2np in zip(model1.named_parameters(), model2.named_parameters()):
		name = m1np[0]
		

In [6]:
#RNN

blackbox_dict_path = "rnn/seed_31_RNNx28_outer_iterations_55_num_samples_10000_num_epochs_5_dataset_mnist_optim_adam_activation_relu_sampling_method_committee_aligner_var_RNN_28/black_box.pt"
blackbox_og_params_dict_path = "rnn/seed_31_RNNx28_outer_iterations_55_num_samples_10000_num_epochs_5_dataset_mnist_optim_adam_activation_relu_sampling_method_committee_aligner_var_RNN_28/original_params_black_box.pt"
final_population_dict_path = "rnn/seed_31_RNNx28_outer_iterations_55_num_samples_10000_num_epochs_5_dataset_mnist_optim_adam_activation_relu_sampling_method_committee_aligner_var_RNN_28/population_iteration_54.pt"

best_model_index = 1 #needs to be manually inspected from log file

blackbox_dict = torch.load(blackbox_dict_path)
blackbox_original_params_dict = torch.load(blackbox_og_params_dict_path)
final_population_dict = torch.load(final_population_dict_path)

blackbox = var_RNN(28, [28]) #input_size, layer_configs
blackbox.load_state_dict(blackbox_dict)

subs = [var_RNN(28, [28]) for i in range(10)]
print(final_population_dict.keys())
final_population = Population(subs=subs)
final_population.load_state_dict(final_population_dict)
final_population.best = best_model_index
best_model = final_population.subs[best_model_index]

final_population.evaluate(blackbox, model_type='rnn')

odict_keys(['subs.0.layers.0.weight_ih_l0', 'subs.0.layers.0.weight_hh_l0', 'subs.0.layers.0.bias_ih_l0', 'subs.0.layers.0.bias_hh_l0', 'subs.0.layers.1.weight', 'subs.0.layers.1.bias', 'subs.1.layers.0.weight_ih_l0', 'subs.1.layers.0.weight_hh_l0', 'subs.1.layers.0.bias_ih_l0', 'subs.1.layers.0.bias_hh_l0', 'subs.1.layers.1.weight', 'subs.1.layers.1.bias', 'subs.2.layers.0.weight_ih_l0', 'subs.2.layers.0.weight_hh_l0', 'subs.2.layers.0.bias_ih_l0', 'subs.2.layers.0.bias_hh_l0', 'subs.2.layers.1.weight', 'subs.2.layers.1.bias', 'subs.3.layers.0.weight_ih_l0', 'subs.3.layers.0.weight_hh_l0', 'subs.3.layers.0.bias_ih_l0', 'subs.3.layers.0.bias_hh_l0', 'subs.3.layers.1.weight', 'subs.3.layers.1.bias', 'subs.4.layers.0.weight_ih_l0', 'subs.4.layers.0.weight_hh_l0', 'subs.4.layers.0.bias_ih_l0', 'subs.4.layers.0.bias_hh_l0', 'subs.4.layers.1.weight', 'subs.4.layers.1.bias', 'subs.5.layers.0.weight_ih_l0', 'subs.5.layers.0.weight_hh_l0', 'subs.5.layers.0.bias_ih_l0', 'subs.5.layers.0.bias_hh

model_type:  rnn


In [7]:
best_model_std = Standardizer(best_model)
blackbox_std = Standardizer(blackbox)

best_model_std.align(blackbox_std)
best_model = best_model_std.reload()
blackbox = blackbox_std.reload()




print("blackbox")
print_weights(blackbox)

blackbox
Layer: layers.0.weight_ih_l0 - Weights
Parameter containing:
tensor([[-0.0143,  0.0583,  0.0029,  0.0950, -0.0279,  0.0952,  0.1227,  0.0469,
         -0.0479, -0.1089, -0.2271, -0.1227, -0.0584, -0.1818,  0.1126,  0.0988,
         -0.2581, -0.3756, -0.2542,  0.0742,  0.1925,  0.1260,  0.2240,  0.3260,
          0.0502,  0.1556,  0.2300,  0.2297],
        [-0.0435, -0.2716, -0.2271, -0.3194, -0.0698, -0.2394,  0.1815,  0.1379,
          0.2998,  0.0730,  0.1542,  0.2894,  0.2078,  0.0767,  0.1427,  0.1913,
          0.2456,  0.1081,  0.1361,  0.0341,  0.0869,  0.2330, -0.1154,  0.0281,
          0.0915, -0.1441, -0.1229, -0.0158],
        [-0.2548, -0.0969, -0.3157, -0.2243, -0.0300, -0.0964, -0.0786,  0.2240,
          0.0188,  0.1428,  0.2869,  0.1038,  0.3151,  0.1059,  0.2531,  0.0887,
          0.1756,  0.1883,  0.1386, -0.0183, -0.1130, -0.0262,  0.0474,  0.0587,
          0.0255,  0.0107, -0.2488, -0.0403],
        [-0.0089, -0.3172, -0.3753, -0.1686, -0.3271, -0.0048, 

In [ ]:
print("reconstruction")
print_weights(best_model)

In [ ]:
y = best_model.layers[0].weight_ih_l0.flatten()
x = blackbox.layers[0].weight_ih_l0.flatten()
print(x.sort()[0] - y.sort()[0] )


y = best_model.layers[0].weight_hh_l0.flatten()
x = blackbox.layers[0].weight_hh_l0.flatten()
print(x.sort()[0] - y.sort()[0] )


y = best_model.layers[1].weight
x = blackbox.layers[1].weight
print(x.sort()[0] - y.sort()[0] )

In [ ]:
y = best_model.layers[0].weight_ih_l0.flatten()
x = blackbox.layers[0].weight_ih_l0.flatten()
print(x-y)


y = best_model.layers[0].weight_hh_l0.flatten()
x = blackbox.layers[0].weight_hh_l0.flatten()
print(x-y)


y = best_model.layers[1].weight
x = blackbox.layers[1].weight
print(x-y)

In [ ]:
#transformer

blackbox_dict_path = "transformer/seed_0_transx64_outer_iterations_55_num_samples_10000_num_epochs_5_dataset_mnist_optim_adam_activation_relu_sampling_method_committee_aligner_test3/black_box.pt"
blackbox_og_params_dict_path = "transformer/seed_0_transx64_outer_iterations_55_num_samples_10000_num_epochs_5_dataset_mnist_optim_adam_activation_relu_sampling_method_committee_aligner_test3/original_params_black_box.pt"
final_population_dict_path = "transformer/seed_0_transx64_outer_iterations_55_num_samples_10000_num_epochs_5_dataset_mnist_optim_adam_activation_relu_sampling_method_committee_aligner_test3/population_iteration_54.pt"

best_model_index = 1

blackbox_dict = torch.load(blackbox_dict_path)
blackbox_original_params_dict = torch.load(blackbox_og_params_dict_path)
final_population_dict = torch.load(final_population_dict_path)

blackbox = base_TransformerEncoder(28, [64]) #d_model, layer_configs (size of linear layer)
blackbox.load_state_dict(blackbox_dict)
 
subs = [base_TransformerEncoder(28, [64]) for i in range(10)]
print(final_population_dict.keys())
final_population = Population(subs=subs)
final_population.load_state_dict(final_population_dict)
final_population.best = best_model_index
best_model = final_population.subs[best_model_index]

final_population.evaluate(blackbox, model_type='trans')